# Module 3 Lab: Degree Sequences and Graph Isomorphisms

In this lab, we will compute the degree sequences of multiple graphs. We will also see some algorithms that attempt to check if two graphs are isomorphic and learn some additional visualization tools in Gephi.

We will begin by building three edge lists (slightly larger than our prior graphs) and convert them to graph objects.

In [ ]:
# import the necessary helpers
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from lab3_helpers import (build_graph_from_edge_list,
                          graph_summary,
                          plot_degree_histogram)

In [ ]:
edges1 = [
        ("A", "B"),
        ("A", "C"),
        ("A", "D"),
        ("B", "C"),
        ("B", "E"),
        ("C", "F"),
        ("D", "E"),
        ("D", "F"),
        ("E", "G"),
        ("F", "H"),
        ("G", "H"),
    ]

In [ ]:
edges2 = [
        (1, 2),
        (1, 3),
        (1, 4),
        (2, 3),
        (2, 5),
        (3, 6),
        (4, 5),
        (4, 6),
        (5, 7),
        (6, 8),
        (7, 8),
    ]

In [ ]:
edges3 = [
        ("A", "B"),
        ("A", "C"),
        ("A", "D"),
        ("B", "C"),
        ("B", "E"),
        ("C", "F"),
        ("D", "E"),
        ("D", "F"),
        ("E", "G"),
        ("F", "G"),
        ("G", "H"),
    ]

In [ ]:
G1 = build_graph_from_edge_list(edges1)
G2 = build_graph_from_edge_list(edges2)
G3 = build_graph_from_edge_list(edges3)

Our first step will be to plot a histogram of the degree to view the distribution.

In [ ]:
plot_degree_histogram(G1)

In [ ]:
# TODO: plot the degree distributions for the other two graphs.
# What do you notice?

In [ ]:
# TODO: export all three graphs to Gephi.
# Load in the graphs and in the left-hand panel, select the size button 
# (looks like three overlapping circles). Click the ranking tab and from the drop down
# select Degree. Play around with some min and max sizes.

Now that we have the three graphs loaded, we can build their degree sequences.
Recall that the degree sequence is USEFUL for determining if there is an isomorphism or not, but it is NOT SUFFICIENT to call two graphs isomorphic only if the degree sequences will match.

In [ ]:
def degree_sequence(_g, _descending=True):
    degrees = [deg for _, deg in _g.degree()]
    return sorted(degrees, reverse=_descending)

In [ ]:
ds1 = degree_sequence(G1)
ds2 = degree_sequence(G2)
ds3 = degree_sequence(G3)

In [ ]:
# TODO: compare the three degree sequences. 
# Do any of them match? Do any mismatch? 
#  If there are mis-matches, what does this tell you about an isomorphism between the graphs?

Next, we will use some isomorphism checking functions from the NetworkX library.
These functions implement algorithms that are optimized for speed and are beyond the complexity of this course.
The documentation for these functions (and references to the papers that developed these algorithms) can be found at https://networkx.org/documentation/stable/reference/algorithms/isomorphism.html.

First, we will use nx.is_isomorphic()

In [ ]:
nx.is_isomorphic(G1, G2)

In [ ]:
nx.is_isomorphic(G1, G3)

In [ ]:
nx.is_isomorphic(G2, G3)

There are two isomorphic graphs: G1 and G2. Take a look at their edge lists and the naming of the vertices. What is different about the two? Can you find the explicit isomorphism between these two graphs?

This function is a nice 'checker' of isomorphism, but finding the isomorphism requires more care and the utilization of proof-based methods.

### Scaling isomorphism

The function also has it's limitations. Since finding graph isomorphisms is hard (in fact, it is part of a class of decision problems called NP - those that can only be solved in nondeterministic polynomial time), computational tricks need to be applied, especially when handling very large graphs. We can look at an example of this by generating some large graphs, which we will be studying more closely in the second half of the course.

In [ ]:
def generate_random_graph(n, m, seed=None):
    """
    Generate a random graph with exactly n nodes and m edges.
    """
    max_edges = n * (n - 1) // 2
    if m < 0 or m > max_edges:
        raise ValueError(f"For a simple undirected graph on {n} nodes, m must be between 0 and {max_edges}.")
    return nx.gnm_random_graph(n, m, seed=seed)

In [ ]:
big_graph_1 = generate_random_graph(1000, 4000, 17)
big_graph_2 = generate_random_graph(1000, 4000, 19)

In [ ]:
nx.is_isomorphic(big_graph_1, big_graph_2)

Even when we have 1000 vertices and 4000 edges, the function runs pretty smoothly.

Let's then consider how this scales with respect to the number of vertices and the number of edges.
We will do this by looping over two lists of numbers: one for large number of vertices, and one for large number of edges. Each pair we will call an 'experiment'. For each experiment, we will record the run time (in seconds) and if the two graphs are isomorphic.


In [ ]:
run_time = []
is_iso = []
n_verts = [40, 60, 80, 100, 120, 140, 160, 180, 200]
n_edges = [2*n for n in n_verts]
tests = [(n_verts[i], n_edges[i]) for i in range(len(n_verts))]
for exp in tests:
        n = int(exp[0])
        e = int(exp[1])
        start = time.time()
        print(f"Generating graphs with {n} vertices and {e} edges")
        bg1 = generate_random_graph(n, e, 17)
        bg2 = generate_random_graph(n, e, 19)
        print("Running isomorphism")
        is_iso.append(nx.is_isomorphic(bg1, bg2))
        run_time.append(time.time() - start)


Let's inspect the results. We can plot the number of vertices versus the run time and the number of edges versus the run time.

In [ ]:
def plot_isomorphism_timings(vertex_sizes, edge_sizes, run_times):
    """
    Make two plots:
      1. vertices vs run time, with a quadratic overlay
      2. edges vs run time, with a quadratic overlay

    The quadratic overlay is a fitted curve of the form a*x^2 + b*x + c.
    """
    x_v = np.array(vertex_sizes, dtype=float)
    x_e = np.array(edge_sizes, dtype=float)
    y = np.array(run_times, dtype=float)

    # Fit quadratic polynomials
    coeffs_v = np.polyfit(x_v, y, 2)
    coeffs_e = np.polyfit(x_e, y, 2)

    poly_v = np.poly1d(coeffs_v)
    poly_e = np.poly1d(coeffs_e)

    x_v_smooth = np.linspace(x_v.min(), x_v.max(), 200)
    x_e_smooth = np.linspace(x_e.min(), x_e.max(), 200)

    plt.figure(figsize=(7, 5))
    plt.plot(x_v, y, marker="o", linestyle="", label="Observed run time")
    plt.plot(x_v_smooth, poly_v(x_v_smooth), label="Quadratic fit")
    plt.xlabel("Number of vertices")
    plt.ylabel("Run time (seconds)")
    plt.title("Vertices vs Isomorphism Run Time")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(x_e, y, marker="o", linestyle="", label="Observed run time")
    plt.plot(x_e_smooth, poly_e(x_e_smooth), label="Quadratic fit")
    plt.xlabel("Number of edges")
    plt.ylabel("Run time (seconds)")
    plt.title("Edges vs Isomorphism Run Time")
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
plot_isomorphism_timings(n_verts, n_edges, run_time)

As we can see, the measurements look roughly quadratic for the number of vertices and edges over this tested range. This tracks with the theory: the complexity of graph isomorphism testing is on the order of O(n^2). What this means is that as the number of vertices n grows, the running time grows roughly like the square of the input size. In plain language, if the input gets twice as large, the time might become about four times as large; if the input gets three times as large, the time might become about nine times as large.
